<a href="https://colab.research.google.com/github/bfs0/SDAIA_Athkax_A-B_Testing_PROJECT/blob/main/Copy_of_%D8%AA%D9%85%D8%B1%D9%8A%D9%86_1_1%D8%8C_%D8%A7%D9%84%D9%85%D9%82%D8%A7%D8%B1%D9%86%D8%A9_%D8%A7%D9%84%D8%B3%D8%A7%D8%B0%D8%AC%D8%A9_%D9%88%D8%A7%D9%84%D9%82%D8%B1%D8%B9%D8%A9.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

<div dir="rtl">

# تمرين 1.1: المقارنة الساذجة والقرعة

**برنامج التجارب واختبارات A/B والاستدلال السببي**، اليوم الأول

**الخطوات:**
1. من قائمة File اختر Save a copy in Drive، واعمل على النسخة التي تُفتح.
2. شغّل الخلايا بالترتيب، ثم اكتب حلك في الخلية المخصصة.
3. اضغط Share، واجعل General access على Anyone with the link بصلاحية Viewer، ثم اضغط Copy link.
4. الصق الرابط في ملف التقييم الخاص بمجموعتك، في عمود «تمرين 1.1».

</div>

<div dir="rtl">

## الإعداد

شغّل الخلية التالية أولًا.

</div>

In [ ]:
import warnings
warnings.filterwarnings("ignore")

import html
from IPython.display import HTML, display

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from cycler import cycler
from matplotlib import ft2font
from scipy import stats

# الإصدارات الحديثة من matplotlib تعرض العربية مباشرة، والقديمة تحتاج مكتبتين صغيرتين
NATIVE_ARABIC = hasattr(ft2font, "__libraqm_version__")
if not NATIVE_ARABIC:
    try:
        import arabic_reshaper
        from bidi.algorithm import get_display
    except ImportError:
        import subprocess
        import sys
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", "arabic-reshaper", "python-bidi"], check=True)
        import arabic_reshaper
        from bidi.algorithm import get_display

rng = np.random.default_rng(2026)

# ألوان البرنامج: A تركوازي، وB برتقالي، والقيمة الحقيقية كحلي، والطريقة الساذجة رمادي
TEAL, ORANGE, NAVY, GRAY = "#0E7490", "#E58A1F", "#0F2A3F", "#A9B4BC"
plt.rcParams.update({
    "figure.figsize": (9, 4.5), "figure.dpi": 110,
    "axes.prop_cycle": cycler(color=[TEAL, ORANGE, NAVY, GRAY]),
    "axes.edgecolor": "#56656F", "axes.labelcolor": NAVY, "text.color": NAVY,
    "xtick.color": "#3D4F5C", "ytick.color": "#3D4F5C",
    "axes.grid": True, "grid.color": "#D8E1E7", "grid.linewidth": 0.8,
    "axes.spines.top": False, "axes.spines.right": False,
    "axes.titlesize": 13, "axes.titleweight": "bold", "font.size": 11,
    "legend.frameon": False,
})
pd.set_option("display.precision", 4)


def ar(text):
    """يجهز النص العربي ليظهر بالشكل والاتجاه الصحيحين داخل الرسوم."""
    text = str(text)
    return text if NATIVE_ARABIC else get_display(arabic_reshaper.reshape(text))


def say(*parts, sep=" "):
    """يطبع نصًا عربيًا من اليمين إلى اليسار."""
    text = html.escape(sep.join(str(p) for p in parts))
    display(HTML(f'<div dir="rtl" style="white-space: pre-wrap; line-height: 1.7;">{text}</div>'))

import hashlib

say("الإعداد جاهز")

<div dir="rtl">

## الأدوات الجاهزة

هذه الدوال والبيانات من دفتر اليوم، ويحتاجها التمرين. شغّلها كما هي.

</div>

In [ ]:
def simulate_world(true_effect=0.0, n=20_000, seed=1):
    """عالم محاكى: التفاعل يرفع استخدام الميزة ويرفع الشراء في الوقت نفسه."""
    r = np.random.default_rng(seed)
    engagement = r.normal(0, 1, n)                                     # مستوى التفاعل
    uses_feature = r.binomial(1, 1 / (1 + np.exp(-1.5 * engagement)))  # الأكثر تفاعلًا يستخدمون الميزة أكثر
    base = 1 / (1 + np.exp(-(-2.2 + 1.0 * engagement)))               # احتمال الشراء بدون الميزة
    u = r.random(n)
    y0 = (u < base).astype(int)                                        # النتيجة إذا لم يستخدم الميزة
    y1 = (u < np.clip(base + true_effect, 0, 1)).astype(int)           # النتيجة إذا استخدمها
    t_random = r.binomial(1, 0.5, n)                                   # قرعة عادلة نستخدمها لاحقًا
    return pd.DataFrame({"engagement": engagement, "uses_feature": uses_feature,
                         "y0": y0, "y1": y1, "t_random": t_random})


world = simulate_world(true_effect=0.0)
say(f"الأثر الحقيقي في هذا العالم: {(world.y1 - world.y0).mean() * 100:.2f} نقطة مئوية")

In [ ]:
def compare_methods(df):
    """نقارن المقارنة الساذجة والتجربة العشوائية بالأثر الحقيقي."""
    true_ate = (df["y1"] - df["y0"]).mean()
    y_obs = np.where(df["uses_feature"] == 1, df["y1"], df["y0"])          # ما نراه في الواقع
    naive = y_obs[df["uses_feature"] == 1].mean() - y_obs[df["uses_feature"] == 0].mean()
    y_rand = np.where(df["t_random"] == 1, df["y1"], df["y0"])             # لو وزعنا الميزة بالقرعة
    randomized = y_rand[df["t_random"] == 1].mean() - y_rand[df["t_random"] == 0].mean()
    return pd.Series({"الأثر الحقيقي (ATE)": true_ate,
                      "المقارنة الساذجة": naive,
                      "التجربة العشوائية": randomized})


(compare_methods(world) * 100).round(2).to_frame("بالنقطة المئوية")

,بالنقطة المئوية
الأثر الحقيقي (ATE),0.00
المقارنة الساذجة,11.02
التجربة العشوائية,0.81


<div dir="rtl">

## المطلوب

اجعل الأثر الحقيقي صفرًا (`true_effect=0.0`) وأعد المقارنة. هل ما زالت المقارنة الساذجة تُظهر أثرًا؟ ماذا تستنتج؟

</div>

In [ ]:
no_effect_world = simulate_world(true_effect=0.0, seed=7)
compare_methods(no_effect_world)

# نعم المقارنه الساذجه لا زالت تظهر اثر
# الاستنتاج : المقارنه الساذجه تظهر قيمه حتى لو ان الاثر الحقيقي بصفر

,0
الأثر الحقيقي (ATE),0.0000
المقارنة الساذجة,0.1114
التجربة العشوائية,-0.0042


<div dir="rtl">

## التسليم

انتهيت؟ اضغط Share، ثم Copy link، والصق الرابط في ملف التقييم في عمود «تمرين 1.1».

</div>